# Quickstart: Pandas API on Spark

https://mybinder.org/v2/gh/apache/spark/master?filepath=python%2Fdocs%2Fsource%2Fgetting_started%2Fquickstart_ps.ipynb

This is a short introduction to pandas API on Spark, geared mainly for new users. This notebook shows you some key differences between pandas and pandas API on Spark. You can run this examples by yourself in 'Live Notebook: pandas API on Spark' at the quickstart page.

Customarily, we import pandas API on Spark as follows:

In [3]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession
import os

In [4]:
os.environ["PYARROW_IGNORE_TIMEZONE"] ='1'

## Object Creation
Creating a pandas-on-Spark Series by passing a list of values, letting pandas API on Spark create a default integer index:

In [5]:
s = ps.Series([1, 3, 5, np.nan, 6, 8])
s

24/03/11 15:55:18 WARN Utils: Your hostname, MacBook-Air-M2.local resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
24/03/11 15:55:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/11 15:55:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
dtype: float64

Creating a pandas-on-Spark DataFrame by passing a dict of objects that can be converted to series-like.

In [6]:
psdf = ps.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])
psdf

,a,b,c
10,1,100,one
20,2,200,two
30,3,300,three
40,4,400,four
50,5,500,five
60,6,600,six


In [6]:
psdf = ps.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])
psdf

,a,b,c
10,1,100,one
20,2,200,two
30,3,300,three
40,4,400,four
50,5,500,five
60,6,600,six


In [6]:
psdf = ps.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])
psdf

,a,b,c
10,1,100,one
20,2,200,two
30,3,300,three
40,4,400,four
50,5,500,five
60,6,600,six


In [6]:
psdf = ps.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])
psdf

,a,b,c
10,1,100,one
20,2,200,two
30,3,300,three
40,4,400,four
50,5,500,five
60,6,600,six


In [7]:
import numpy as np
from math import ceil

from skimage.color import rgb2gray
from skimage.filters import sobel
from skimage.segmentation import slic
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float

import imageio.v2 as imageio
from skimage.measure import regionprops, regionprops_table
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import SparkSession

import matplotlib.pyplot as plt
import plotly.express as px
import psutil
import time
import sys
import os

from itertools import product

import pickle
import h5py
import random
from tqdm import tqdm

seed = random.seed(999) # para gerar sempre com a mesma seed



In [11]:
def gen_files_to_read(name_img, bands, read_dir):
    ''''
    gen array with tif files name to read
    '''
    files_to_read = []
    for b in bands:
        files_to_read.append(read_dir+name_img+b+'.tif')
    
    return files_to_read

def load_image_files2(files_name,pos=-2):
    ''''
    load tiff image bands files of a timestamp
    '''
    #files_name=[file_nbr,file_evi, file_ndvi, file_red,file_green,file_blue]
    bands_name =[]
    image_band_dic={}
    for f in files_name:
        f_name = f.split('.')  #para remover o .tif
        #print ('f_name:',f_name)
        band = f_name[0].split("_") # para obter onome da banda do arquivo
        #print (band[pos])
        bands_name.append(band[pos])
        image_band_dic[band[pos]] = imageio.imread(f)

    return image_band_dic


In [12]:
read_dir = '/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/data/Cassio/S2-16D_V2_012014_20220728_/'
name_img = 'S2-16D_V2_012014_20220728_'

current_directory = os.getcwd()
#for new image 23/02/2024
upper_level_directory = os.path.join(current_directory, '../data/test_segm_results/S2-16D_V2_012014_20220728_/')

# Specify the filename and path within the upper-level directory
#filename = 'SENTINEL-2_MSI_20LMR_RGB_2022-07-16'
#for new image
save_path = os.path.join(upper_level_directory, name_img)

all_bands = ['B04', 'B03', 'B02', 'B08', 'EVI', 'NDVI']
bands_sel = ['B04', 'B03', 'B02'] # R,G,B bands selection for slic segmentation

# generate an array with tiff files name to be loaded
tif_names=[]
tif_names=gen_files_to_read(name_img, all_bands, read_dir)
#%%
time_ini= time.time()
image_band_dic = {}
image_band_dic = load_image_files2(tif_names, pos=-1)
time_band = time.time()
#%%
time_proc ={}
time_proc['load_band_files'] = time_band - time_ini

time_ini = time.time()
img_sel = np.dstack((image_band_dic[bands_sel[0]], image_band_dic[bands_sel[1]], image_band_dic[bands_sel[2]]))
time_fim = time.time()
time_proc['gen_img_rgb'] = time_fim - time_ini

# gen mask
mask=None
contains_nan={}

# Check if the matrix contains the element -9999
time_ini = time.time()
for i,b  in enumerate(bands_sel):
# Check if the matrix contains the element -9999
    contains_nan[i] = np.any(image_band_dic[bands_sel[i]] == -9999)
    if contains_nan[i]:
        #print ("true")
        if i==0:
            mask = (image_band_dic[bands_sel[i]] != -9999) 
        else:
            mask = (image_band_dic[bands_sel[i]] != -9999) & mask
time_mask= time.time()

del contains_nan
time_proc['gen_mask']=time_mask-time_ini

#%%
#getting the image with values normalized
time_ini=time.time()
image_band_dic_norm={}
for k in image_band_dic.keys():
    image_band_dic_norm[k]=image_band_dic[k].astype(float)/np.max(image_band_dic[k])
time_norm=time.time()

img_sel_norm = np.dstack((image_band_dic_norm[bands_sel[0]], image_band_dic_norm[bands_sel[1]], image_band_dic_norm[bands_sel[2]]))
time_norm=time.time()
time_proc['gen_img_norm'] = time_norm-time_ini
segms = [110000]
compactness = [1, 2]
sigmas = [0.1, 0.5]
connectivity = [False]
# making all possible parameters combinations 
parameter_combinations = list(product(segms, compactness, sigmas, connectivity))
#%%
params_test_dic = {}
for id,comb in enumerate(parameter_combinations, start=1):
    if id not in params_test_dic:
        params_test_dic[id] = {}
    params_test_dic[id]['segms'] = comb[0]
    params_test_dic[id]['compactness'] = comb[1]
    params_test_dic[id]['sigma'] = comb[2]
    params_test_dic[id]['connectivity'] = comb[3]

#%%    
#running tests for all combinations
ids = list(params_test_dic.keys())
stats_df_dic={}
props_df_sel = {}
props_dic_SP={} 
segments_slic_sel ={}
#%%    
ids=[3]
time_slic={}

In [14]:
n_segms = params_test_dic[id]['segms'] #600
sigma = params_test_dic[id]['sigma']  #0.1
compact = params_test_dic[id]['compactness']
conectivity= params_test_dic[id]['connectivity'] #False
print ("test and params id: ",id, n_segms, sigma, compact, conectivity)


test and params id:  4 110000 0.5 2 False


In [15]:
segments_slic_sel2 = slic(img_sel_norm, n_segments=n_segms, compactness=compact, sigma=sigma, \
                             start_label=1,mask=mask, enforce_connectivity=conectivity)

In [17]:
props_dic = regionprops_table(segments_slic_sel2, img_sel_norm, \
                                properties=['label','centroid','local_centroid']) 

In [64]:
props_psdf = ps.DataFrame(props_dic)

In [68]:
props_psdf.insert(3, "B04", 4)

In [103]:
props_psdf["b05"]=props_psdf["centroid-0"]
def round_ps(x):
    return x, round(13.5)
#props_psdf["b05"] = props_psdf[['centroid-0', 'centroid-1']].apply(lambda row: round_ps(row))
props_psdf["b05"] = props_psdf['centroid-0'].apply(lambda row: round_ps(row))

PySparkTypeError: [CANNOT_ACCEPT_OBJECT_IN_TYPE] `DoubleType()` can not accept object `14` in type `int`.

In [98]:
props_psdf.head()

,label,centroid-0,centroid-1,B04,local_centroid-0,local_centroid-1,b05
0,1,13.903518,16.201005,4,13.903518,16.201005,"[14, 14]"
1,2,16.379893,49.754448,4,16.379893,18.754448,"[16, 14]"
2,3,15.683630,84.742154,4,15.683630,17.742154,"[16, 14]"
3,4,12.545455,121.755892,4,12.545455,22.755892,"[13, 14]"
4,5,15.450980,141.712418,4,15.450980,19.712418,"[15, 14]"


In [22]:
props_dic_SP = regionprops_table(segments_slic_sel2, img_sel_norm, \
                                properties=['label','coords'])   

In [43]:
type(props_psdf)

pyspark.pandas.frame.DataFrame

In [44]:
from pyspark.sql.functions import udf,sum
from pyspark.sql.types import IntegerType

def round_xy(x,y):
    #return round(data[1])+round(data[0])
    return round(x), round(y)
round_xy_udf= udf(round_xy, IntegerType())

for b in image_band_dic.keys():
    print (b)
    #props_psdf[b] = props_psdf.apply(lambda row: image_band_dic[b][round(row['centroid-0']), round(row['centroid-1'])], axis=1)
    #props_psdf[b] = props_psdf.apply(image_band_dic[b][round_xy(col('centroid-1'), col('centroid-0'))], axis=1)
    #props_psdf.withColumn(b, props_psdf.apply(round_xy_udf, axis=1))
    props_psdf= props_psdf.withColumn(b, lit(10))#sum(props_psdf['centroid-0'],props_psdf['centroid-1']))

B04


AttributeError: 'DataFrame' object has no attribute 'withColumn'

In [26]:
import pyspark.pandas as ps
import numpy as np

technologies = ({
    'Fee' :[20000,25000,30000,22000,np.NaN],
    'Discount':[1000,2500,1500,1200,3000]
               })
# Create a DataFrame
psdf = ps.DataFrame(technologies)
print(psdf)

def add(data):
   return data[0] + data[1]
   
addDF = psdf.apply(add,axis=1)
print(addDF)

       Fee  Discount
0  20000.0      1000
1  25000.0      2500
2  30000.0      1500
3  22000.0      1200
4      NaN      3000


/Users/flaviaschneider/Documents/flavia/Data_GEOBIA/.venv_data_geobia/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0    21000.0
1    27500.0
2    31500.0
3    23200.0
4        NaN
dtype: float64
